# MCO como problema de optimización

**¿Podemos encontrar los estimadores de MCO sin usar una función de regresión?**

MCO elige los valores de $b_1$ y $b_2$ que minimizan la suma de cuadrados de los residuos. En este notebook pediremos a R que busque numéricamente esos valores.

[Descargar este notebook (.ipynb)](https://github.com/amosino/mtx--courses/raw/refs/heads/main/MTX1/notebooks/01--mco_optimizacion.ipynb)


## Importamos los datos

Usamos `read_csv()` del paquete `readr` para cargar la base desde una ruta relativa. Después mostramos las primeras observaciones para verificar que las variables que utilizaremos están disponibles y fueron leídas correctamente.

In [1]:
library(readr)

datos <- read_csv("../data/lexpectancy.csv")
head(datos)

Rows: 223 Columns: 3


── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): pais
dbl (2): rgdp, life_exp



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


pais,rgdp,life_exp
<chr>,<dbl>,<dbl>
Afghanistan,2000,51.3
Albania,11900,78.3
Algeria,15000,76.8
American Samoa,13000,75.4
Andorra,37200,82.8
Angola,6800,56.0


## Elegimos un punto de partida

`optim()` necesita valores iniciales para comenzar la búsqueda. Estos valores no son los estimadores: son solamente el punto de partida del procedimiento. También extraemos las variables que forman la regresión: esperanza de vida ($y$) y producto por habitante ($x$).

In [2]:
b1_0 <- 60
b2_0 <- 0.05

y <- datos$life_exp
x <- datos$rgdp

## Definimos la función objetivo

Queremos encontrar los valores de $b_1$ y $b_2$ que minimizan la suma de cuadrados de los residuos.

Por eso definimos una **función objetivo** que depende precisamente de los parámetros que estamos buscando. Para cada par candidato $(b_1,b_2)$, la función:

1. construye los valores estimados $\hat y_i=b_1+b_2x_i$;
2. calcula los residuos $e_i=y_i-\hat y_i$;
3. devuelve su suma de cuadrados:

$$
RSS(b_1,b_2)=\sum_{i=1}^{n}(y_i-b_1-b_2x_i)^2.
$$

El problema de MCO consiste entonces en encontrar el par $(b_1,b_2)$ que hace mínima esta función.

En `params` guardamos los valores candidatos de $b_1$ y $b_2$.

In [3]:
rss <- function(params) {
  b1 <- params[1]
  b2 <- params[2]

  y_hat <- b1 + b2 * x
  e <- y - y_hat

  sum(e^2)
}

## Buscamos el mínimo

`optim()` prueba distintos pares de valores y busca el par que hace mínima la función `rss()`. `par` contiene el punto de partida y `fn` indica la función que queremos minimizar.

In [4]:
mco <- optim(
  par = c(b1_0, b2_0),
  fn = rss,
  method = "BFGS"
)

## Examinamos la solución

`mco$par` contiene los valores de $b_1$ y $b_2$ encontrados por el procedimiento. `mco$value` muestra la suma de cuadrados de los residuos en ese punto.

In [5]:
mco$par
mco$value

[1] 6.729933e+01 2.375081e-04

[1] 9511.585

## Interpretación

Los valores obtenidos son los coeficientes que minimizan la suma de cuadrados de los residuos para esta muestra. Así, hemos encontrado los estimadores de MCO sin utilizar una función de regresión.

En el siguiente notebook veremos que `lm()` obtiene directamente esta misma solución.